# OZ-PERF-002 catalogue quality priority analysis

This bounded notebook reproduces the GSC aggregates used to rank the next catalogue evidence queue. It is audit-only: it does not change indexation, redirects, metadata, schema, catalogue data, or page copy. Positions are impression-weighted within each explicitly selected cohort.

In [1]:
from pathlib import Path
import csv, hashlib, json, re

ROOT = Path.cwd()
GSC = ROOT / 'docs/seo-migration/generated/gsc-equity-map-2026-09-01.csv'
QUALITY = ROOT / 'docs/seo-migration/generated/catalogue-quality-report.json'
HARDENING = ROOT / 'docs/seo-migration/generated/seo-migration-hardening-report.json'
MANUAL = ROOT / 'docs/seo-migration/generated/catalogue-equity-manual-review-2026-09-01.csv'
sources = [GSC, QUALITY, HARDENING, MANUAL, ROOT / '_redirects', ROOT / 'data/product-catalogue.json']
source_sha256 = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in sources}
source_sha256

{'docs/seo-migration/generated/gsc-equity-map-2026-09-01.csv': '1bcf1c50d772ac5b712441a726d7842a5d71518c0ddb4969f07baf2ab5026457', 'docs/seo-migration/generated/catalogue-quality-report.json': '58a3740e71194d17e07be706078fd986aecb203957331fbcea0af3020623901c', 'docs/seo-migration/generated/seo-migration-hardening-report.json': 'a7fe84bd8b3dc96a14a94e4de021a5117343864634f03c30f79f8c9f3109e635', 'docs/seo-migration/generated/catalogue-equity-manual-review-2026-09-01.csv': '6d69cb3c4c9c3f5f408e4b54116c2828c72d44be798a78049965c54022dd859f', '_redirects': 'd40d78e4ceecc6de30302c9a30e689504da9e4bbb4caddd1c38e3f939c27ac84', 'data/product-catalogue.json': '52621e05afcaa88b1c87d947db45ecebc57b6179688a3b2f6216a8594e235e55'}

In [2]:
with GSC.open(encoding='utf-8-sig', newline='') as handle:
    rows = list(csv.DictReader(handle))

selectors = {
  1: r'^/product/solid-raw-timber-',
  2: r'grand.?oak',
  3: r'^/product/krono?swiss-aquastop-',
  4: r'^/product/stone-floor-',
  5: r'^/product-category/solid-timber/topdeck-pre-finished-solid-timber/$',
  6: r'^/product/storm-',
  7: r'^/product/kempas-',
  8: r'^/product/jatoba-brazilian-cherry/$',
  9: r'infinite',
 10: r'^/product/verdura-french-bleed/$',
 11: r'^/product/(?:kerangi-king-teak|ornato-hybrid-oak-sofia|luxury-hybrid-vienna|easi-plank-blackbutt)/$',
 12: r'^/product/hydroplank-',
 13: r'^/product/prestige-oak-(?:hex-grey|derby-brown)/$',
 14: r'pronto',
 15: r'^/product/preference-select-',
 16: r'aspire',
 17: r'^/product/swish-blackbutt/$',
 18: r'^/product/sydney-blue-gum/$',
 19: r'^/product/villeroy-boch-aquastop-sand-oak/$',
 20: r'^/product/etf-hybrid-spc-9mm-helena-oak/$',
 21: r'^/product/(?:swish-oak-fiano-brown|swish-oak-raw-caramel)/$',
}
expected = {
 1:(11,18,5079,46.79), 2:(6,14,1090,33.78), 3:(6,14,484,22.55),
 4:(6,10,1521,10.69), 5:(1,9,901,36.71), 6:(3,8,225,15.44),
 7:(2,7,901,18.26), 8:(1,7,330,27.18), 9:(2,7,264,19.94),
 10:(1,7,128,25.52), 11:(4,5,88,13.07), 12:(4,5,28,10.82),
 13:(2,4,46,13.06), 14:(3,3,705,28.66), 15:(2,3,43,14.23),
 16:(3,3,40,20.17), 17:(2,3,13,6.08), 18:(1,2,638,74.83),
 19:(1,2,90,55.14), 20:(1,2,51,6.08), 21:(2,2,8,5.00),
}

def aggregate(pattern):
    selected = [row for row in rows if re.search(pattern, row['old_path'], re.I)]
    clicks = sum(float(row['clicks']) for row in selected)
    impressions = sum(float(row['impressions']) for row in selected)
    weighted_position = sum(float(row['average_position']) * float(row['impressions']) for row in selected) / impressions
    return (len(selected), int(clicks), int(impressions), round(weighted_position, 2))

aggregates = {rank: aggregate(pattern) for rank, pattern in selectors.items()}
assert aggregates == expected
[{'rank': rank, 'rows': values[0], 'clicks': values[1], 'impressions': values[2], 'weighted_position': values[3]} for rank, values in aggregates.items()]

[{'rank': 1, 'rows': 11, 'clicks': 18, 'impressions': 5079, 'weighted_position': 46.79}, {'rank': 2, 'rows': 6, 'clicks': 14, 'impressions': 1090, 'weighted_position': 33.78}, {'rank': 3, 'rows': 6, 'clicks': 14, 'impressions': 484, 'weighted_position': 22.55}, {'rank': 4, 'rows': 6, 'clicks': 10, 'impressions': 1521, 'weighted_position': 10.69}, {'rank': 5, 'rows': 1, 'clicks': 9, 'impressions': 901, 'weighted_position': 36.71}, {'rank': 6, 'rows': 3, 'clicks': 8, 'impressions': 225, 'weighted_position': 15.44}, {'rank': 7, 'rows': 2, 'clicks': 7, 'impressions': 901, 'weighted_position': 18.26}, {'rank': 8, 'rows': 1, 'clicks': 7, 'impressions': 330, 'weighted_position': 27.18}, {'rank': 9, 'rows': 2, 'clicks': 7, 'impressions': 264, 'weighted_position': 19.94}, {'rank': 10, 'rows': 1, 'clicks': 7, 'impressions': 128, 'weighted_position': 25.52}, {'rank': 11, 'rows': 4, 'clicks': 5, 'impressions': 88, 'weighted_position': 13.07}, {'rank': 12, 'rows': 4, 'clicks': 5, 'impressions': 28,

In [3]:
quality = json.loads(QUALITY.read_text())
hardening = json.loads(HARDENING.read_text())
issue_counts = {}
for finding in hardening['qualityFindings']:
    for issue in finding['issues']:
        issue_counts[issue] = issue_counts.get(issue, 0) + 1
missing_source = sum(1 for page in quality['pages'] if any(issue['code'] == 'missing-source-record' for issue in page['issues']))
assert issue_counts == {'repeated-placeholder-specs': 347, 'duplicated-name-token': 10, 'image-to-confirm': 5}
assert missing_source == 478
assert len(hardening['qualityFindings']) == 361
{'gsc_rows': len(rows), 'hardening_findings': len(hardening['qualityFindings']), 'issue_counts': issue_counts, 'missing_source_controlled_pages': missing_source}

{'gsc_rows': 130, 'hardening_findings': 361, 'issue_counts': {'repeated-placeholder-specs': 347, 'duplicated-name-token': 10, 'image-to-confirm': 5}, 'missing_source_controlled_pages': 478}